# Phase 2: Focused Exploratory Data Analysis

This notebook explores valid patterns, anomalies, and follow-up questions in the cleaned e-commerce datasets. Amazon and international sales are kept separate; reported sales are not treated as profit; unsupported customer, true return, and inventory-turnover KPIs are excluded.

## 1. Analytical objective

Assess sales coverage, status mix, product/category concentration, reference pricing context, fulfilment mix, stock-snapshot signals, distributions, and anomalies for later investigation. This is descriptive decision support, not causal analysis.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists(): ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.visualizations import apply_business_style
apply_business_style()
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')


## 2. Dataset loading

In [ ]:
CLEAN_DIR = ROOT / 'data' / 'cleaned'
paths = {path.stem: path for path in sorted(CLEAN_DIR.glob('*.csv'))}
datasets = {name: pd.read_csv(path) for name, path in paths.items()}
expected = {'amazon_sale_report_cleaned': {'order_id','date','status','sku','qty','amount'}, 'international_sale_report_cleaned': {'date','months','sku','pcs','rate','gross_amt'}, 'may_2022_cleaned': {'sku','tp','amazon_mrp'}, 'p_l_march_2021_cleaned': {'sku','tp_1','tp_2','amazon_mrp'}, 'sale_report_cleaned': {'sku_code','stock'}, 'cloud_warehouse_compersion_chart_cleaned': {'cost_head','shiprocket_price_per_unit','increff_price_per_unit'}, 'expense_iigf_cleaned': {'transaction_type','record_type','particular','amount'}}
assert set(expected).issubset(datasets)
for name, fields in expected.items(): assert fields.issubset(datasets[name].columns), (name, fields - set(datasets[name].columns))
amazon = datasets['amazon_sale_report_cleaned'].copy()
international = datasets['international_sale_report_cleaned'].copy()
stock = datasets['sale_report_cleaned'].copy()
products_may = datasets['may_2022_cleaned'].copy()
products_march = datasets['p_l_march_2021_cleaned'].copy()
amazon['date'] = pd.to_datetime(amazon['date'], format='%Y-%m-%d', errors='coerce')
international['date'] = pd.to_datetime(international['date'], format='%m/%d/%Y', errors='coerce')
amazon['month_period'] = amazon['date'].dt.to_period('M')
international['month_period'] = international['date'].dt.to_period('M')
print({name: frame.shape for name, frame in datasets.items()})


## 3. Data validation summary

In [ ]:
validation_summary = pd.DataFrame([{'dataset': name, 'rows': len(frame), 'columns': frame.shape[1], 'duplicate_rows': int(frame.duplicated().sum()), 'missing_cells': int(frame.isna().sum().sum())} for name, frame in datasets.items()])
display(validation_summary)
assert validation_summary['duplicate_rows'].sum() == 0
assert amazon['date'].notna().all() and international['date'].notna().all()
assert (international['date'].dt.month == pd.to_datetime(international['months'], format='%b-%y').dt.month).all()
for frame, fields in [(amazon, ['qty','amount']), (international, ['pcs','gross_amt'])]: assert all(pd.api.types.is_numeric_dtype(frame[field]) for field in fields)
print(f'Phase 0 validation status: PASS WITH WARNINGS; dates and numeric fields are usable for scoped EDA. Amazon has {int(amazon.amount.isna().sum()):,} missing amount values ({amazon.amount.notna().mean():.1%} populated); missing values are retained.')


## 4. Dataset grain and coverage

In [ ]:
coverage = pd.DataFrame([['Amazon sales','line',len(amazon),amazon['order_id'].nunique(),amazon['date'].min().date(),amazon['date'].max().date(),amazon['sku'].notna().mean()], ['International sales','line',len(international),np.nan,international['date'].min().date(),international['date'].max().date(),international['sku'].notna().mean()], ['May product prices','SKU/size snapshot',len(products_may),products_may['sku'].nunique(),None,None,1.0], ['March product prices','SKU/size snapshot',len(products_march),products_march['sku'].nunique(),None,None,1.0], ['Stock report','SKU/size/colour snapshot',len(stock),stock['sku_code'].nunique(),None,None,stock['sku_code'].notna().mean()]], columns=['dataset','grain','rows','distinct_key','date_min','date_max','key_coverage'])
display(coverage)
print('Amazon order count uses distinct order_id; line count is not presented as order count.')


## 5. Numerical distributions

**Business question:** What does the scale and shape of reported monetary and unit fields look like?

**Observation:** Amazon amount/quantity, international gross amount/pieces, and stock values are right-skewed with zero or missing values in some fields.

**Interpretation:** Large records may drive totals and should be investigated by SKU/status rather than removed.

**Limitation:** Sources have different currency and scope definitions; this is not a combined revenue distribution.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16,4))
amazon['amount'].dropna().plot.hist(ax=axes[0], bins=40, color='#2f6690', edgecolor='white')
axes[0].set_title('Amazon reported amount', loc='left', fontweight='bold'); axes[0].set_xlabel('Reported amount'); axes[0].set_ylabel('Lines')
international['gross_amt'].dropna().plot.hist(ax=axes[1], bins=40, color='#d1495b', edgecolor='white')
axes[1].set_title('International gross amount', loc='left', fontweight='bold'); axes[1].set_xlabel('Gross amount'); axes[1].set_ylabel('Lines')
stock['stock'].dropna().plot.hist(ax=axes[2], bins=40, color='#3a7d44', edgecolor='white')
axes[2].set_title('Stock snapshot', loc='left', fontweight='bold'); axes[2].set_xlabel('Reported stock units'); axes[2].set_ylabel('Rows')
plt.tight_layout(); plt.show()


## 6. Categorical distributions

**Business question:** Which statuses and merchandise categories dominate?

**Observation:** Amazon status/category distributions are concentrated, while the stock report uses a different category taxonomy.

**Interpretation:** Concentration identifies later deep-dive priorities, not causes.

**Limitation:** Counts are lines or snapshot rows, not orders unless explicitly labelled.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15,5))
amazon['status'].value_counts().head(12).sort_values().plot.barh(ax=axes[0], color='#7b2cbf')
axes[0].set_title('Amazon order-line status mix', loc='left', fontweight='bold'); axes[0].set_xlabel('Lines'); axes[0].set_ylabel('Status')
amazon['category'].value_counts().head(10).sort_values().plot.barh(ax=axes[1], color='#3a7d44')
axes[1].set_title('Amazon top categories by line count', loc='left', fontweight='bold'); axes[1].set_xlabel('Lines'); axes[1].set_ylabel('Category')
plt.tight_layout(); plt.show()


## 7. Date coverage and time trends

**Business question:** How does reported sales activity vary by calendar month within each source?

**Observation:** Amazon covers 91 observed dates from 2022-03-31 to 2022-06-29; March and June are partial boundary months. International sales cover 2021-06-05 to 2022-05-11 with partial June 2021 and May 2022 boundaries. The windows do not fully overlap.

**Interpretation:** Trends must be read within each source window.

**Limitation:** Date-only fields do not establish seasonality or causation.

In [ ]:
amazon_monthly = amazon.groupby('month_period').agg(reported_amount=('amount','sum'), amount_lines=('amount','count'), line_count=('amount','size'), distinct_orders=('order_id','nunique'))
international_monthly = international.groupby('month_period').agg(gross_amount=('gross_amt','sum'), pieces=('pcs','sum'))
fig, axes = plt.subplots(1, 2, figsize=(15,5))
amazon_monthly['reported_amount'].plot.line(ax=axes[0], marker='o', color='#2f6690')
axes[0].set_title('Amazon reported amount by month', loc='left', fontweight='bold'); axes[0].set_xlabel('Month'); axes[0].set_ylabel('Reported amount')
international_monthly['gross_amount'].plot.line(ax=axes[1], marker='o', color='#d1495b')
axes[1].set_title('International gross amount by month', loc='left', fontweight='bold'); axes[1].set_xlabel('Month'); axes[1].set_ylabel('Gross amount')
plt.tight_layout(); plt.show()
amazon_monthly['amount_coverage_pct'] = 100 * amazon_monthly['amount_lines'] / amazon_monthly['line_count']
print(f'Amazon observed dates: {amazon.date.dt.normalize().nunique()}')
print(f'Amazon amount coverage: {amazon.amount.notna().mean():.1%}; missing values are not imputed.')
display(amazon_monthly, international_monthly)


## 8. Order-status analysis

**Business question:** What status mix is visible, and how many orders contain cancellation or return-related labels?

**Observation:** Amazon includes cancelled, returned-to-seller, pending, and delivery-progress labels.

**Interpretation:** These labels support lifecycle monitoring and follow-up questions.

**Limitation:** The figures are status proxies under an analytical convention; multiple lines/statuses can exist within one order, so true rates require a business-approved precedence rule.

In [ ]:
order_flags = amazon.groupby('order_id').agg(has_cancelled=('status', lambda s: s.astype(str).str.contains('Cancelled', case=False, na=False).any()), has_returned=('status', lambda s: s.astype(str).str.contains('Returned|Returning', case=False, regex=True, na=False).any()), has_delivered=('status', lambda s: s.astype(str).str.contains('Delivered', case=False, na=False).any()))
status_proxy = pd.Series({'cancelled-status orders': order_flags['has_cancelled'].sum(), 'returned-status orders': order_flags['has_returned'].sum(), 'delivered-status orders': order_flags['has_delivered'].sum()})
display(status_proxy.to_frame('distinct orders with any matching line status'))
amazon['status'].value_counts().sort_values().plot.barh(color='#7b2cbf')
plt.title('Amazon order-line status mix', loc='left', fontweight='bold'); plt.xlabel('Lines'); plt.ylabel('Status'); plt.show()


## 9. SKU and category coverage

**Business question:** How much sales activity can be attributed to SKU-level product references?

**Observation:** Amazon SKU coverage is higher than international SKU coverage; product snapshot SKUs are unique, but cross-file match gaps remain.

**Interpretation:** Within-source SKU analysis is feasible; enrichment requires match-rate reporting.

**Limitation:** Missing or unmatched SKUs can bias product rankings.

In [ ]:
sku_coverage = pd.DataFrame({'nonmissing_sku_rate':[amazon['sku'].notna().mean(), international['sku'].notna().mean(), stock['sku_code'].notna().mean()], 'distinct_sku':[amazon['sku'].nunique(), international['sku'].nunique(), stock['sku_code'].nunique()]}, index=['Amazon sales','International sales','Stock snapshot'])
display(sku_coverage)
amazon_sku_match = amazon['sku'].dropna().isin(set(products_may['sku'])).mean()
international_sku_match = international['sku'].dropna().isin(set(products_may['sku'])).mean()
print(f'Nonmissing SKU match to May snapshot: Amazon={amazon_sku_match:.1%}; International={international_sku_match:.1%}.')
amazon.groupby('category')['amount'].sum().sort_values(ascending=False).head(8).sort_values().plot.barh(color='#2f6690')
plt.title('Amazon reported amount by top category', loc='left', fontweight='bold'); plt.xlabel('Reported amount'); plt.ylabel('Category'); plt.show()


## 10. Platform or channel coverage

**Business question:** What channel and fulfilment labels are represented in the Amazon extract?

**Observation:** Amazon.in dominates the channel field, with a small Non-Amazon label; fulfilment is split between Amazon and Merchant.

**Interpretation:** Channel and fulfilment mix can be described within this file.

**Limitation:** This is not a complete multi-platform comparison.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,4))
amazon['sales_channel'].value_counts(dropna=False).plot.bar(ax=axes[0], color='#486581')
axes[0].set_title('Amazon extract channel labels', loc='left', fontweight='bold'); axes[0].set_xlabel('Sales channel'); axes[0].set_ylabel('Lines')
amazon['fulfilment'].value_counts(dropna=False).plot.bar(ax=axes[1], color='#3a7d44')
axes[1].set_title('Fulfilment labels', loc='left', fontweight='bold'); axes[1].set_xlabel('Fulfilment'); axes[1].set_ylabel('Lines')
plt.tight_layout(); plt.show()


## 11. Fulfilment analysis

**Business question:** What operational status signals are available for follow-up?

**Observation:** Courier status includes shipped, unshipped, and cancelled labels; provider fields are incomplete.

**Interpretation:** These fields support a descriptive operational queue/mix analysis.

**Limitation:** No timestamps or SLA definitions exist, so delivery-time and on-time KPIs are excluded.

In [ ]:
amazon['courier_status'].value_counts(dropna=False).head(10).sort_values().plot.barh(color='#d1495b')
plt.title('Amazon courier-status mix', loc='left', fontweight='bold'); plt.xlabel('Lines'); plt.ylabel('Courier status'); plt.show()
print('Missing fulfilled_by rate:', f"{amazon['fulfilled_by'].isna().mean():.1%}")


## 12. Outlier investigation

**Business question:** Which numerical records are unusually high under a transparent screening rule?

**Observation:** IQR screening identifies extreme amount, rate, and stock observations.

**Interpretation:** These records are investigation candidates, such as bulk/wholesale lines, shipping lines, or data-entry issues.

**Limitation:** Outliers are retained; the IQR rule is a flag, not evidence of error.

In [ ]:
def iqr_flags(series):
    clean = series.dropna()
    q1, q3 = clean.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return pd.Series({'q1': q1, 'q3': q3, 'lower_bound': lower, 'upper_bound': upper, 'flagged_rows': int(((clean < lower) | (clean > upper)).sum())})
outlier_summary = pd.DataFrame({'amazon_amount': iqr_flags(amazon['amount']), 'international_rate': iqr_flags(international['rate']), 'stock': iqr_flags(stock['stock'])}).T
display(outlier_summary)
display(amazon.nlargest(10, 'amount')[['date','order_id','sku','status','qty','amount']])


## 13. Preliminary relationships

**Business question:** Do monthly reported amounts move with distinct Amazon order counts at the same month grain?

**Observation:** Monthly amount, distinct orders, line count, and units can be compared directly.

**Interpretation:** Divergence can generate questions about mix, order size, or status composition.

**Limitation:** This descriptive comparison does not establish causation, and order-level amount treatment requires source confirmation.

In [ ]:
monthly_relationship = amazon.groupby('month_period').agg(reported_amount=('amount','sum'), distinct_orders=('order_id','nunique'), line_count=('order_id','size'), units=('qty','sum'))
display(monthly_relationship)
assert (monthly_relationship['line_count'] >= monthly_relationship['distinct_orders']).all()
monthly_relationship[['reported_amount','distinct_orders']].plot(secondary_y='distinct_orders', marker='o', figsize=(10,4))
plt.title('Amazon monthly reported amount and distinct orders', loc='left', fontweight='bold'); plt.xlabel('Month'); plt.show()


## 14. Key findings

In [ ]:
amazon_orders = amazon['order_id'].nunique()
amazon_amount = amazon['amount'].sum()
international_gross = international['gross_amt'].sum()
findings = [f'Amazon contains {len(amazon):,} order lines and {amazon_orders:,} distinct orders from {amazon.date.min():%Y-%m-%d} to {amazon.date.max():%Y-%m-%d}.', f'Amazon reported amount totals {amazon_amount:,.2f}; {amazon.amount.notna().mean():.1%} of lines have amount populated and INR is the only populated currency.', f'International sales contain {len(international):,} retained lines and reported gross amount of {international_gross:,.2f}; currency is not supplied, so this remains a separate scope.', f'Amazon has {amazon.qty.sum():,.0f} reported units, including {int((amazon.qty == 0).sum()):,} zero-quantity lines needing status/business review.', f'SKU coverage is {amazon.sku.notna().mean():.1%} for Amazon and {international.sku.notna().mean():.1%} for international sales; cross-source matches are incomplete.', f'The stock snapshot contains {int((stock.stock == 0).sum()):,} zero-stock rows, but no snapshot date, so turnover and stockout-over-time KPIs remain unsupported.']
for finding in findings: print('-', finding)
assert abs(amazon_amount - amazon.groupby('month_period').amount.sum().sum()) < 0.01
assert amazon_orders == amazon.groupby('month_period').order_id.nunique().sum()
assert abs(international_gross - international.groupby('month_period').gross_amt.sum().sum()) < 0.01
report = '# E-commerce EDA Findings\n\n## Key findings\n\n' + '\n'.join(f'- {finding}' for finding in findings) + '\n\n## Limitations\n\n- Amazon and international scopes remain separate because currency and order-grain definitions differ.\n- Status labels are descriptive proxies; true cancellation/return rates require an approved order-level rule.\n- Profit, gross margin, customer-360, true net sales, and inventory turnover are not calculated.\n- Outliers are retained and flagged for follow-up; no causal conclusion is made.\n'
(ROOT / 'reports' / 'eda_findings.md').write_text(report, encoding='utf-8')


## 15. Questions for later phases

- Which Amazon statuses define a business-approved completed order and net-sales scope?
- Are high-amount lines bulk, shipping, or data-entry records?
- Why do many sales SKUs fail to match product snapshots, and can a master SKU mapping be supplied?
- Can dated inventory snapshots and a cost-of-goods ledger be obtained?
- Can international sales receive a currency and order identifier for reconciliation?

## 16. Limitations

- No profit, gross-margin, true net-sales, true return-rate, customer-360, or inventory-turnover KPI is calculated.
- Amazon and international sales are not combined.
- Order counts use distinct Amazon `order_id`; line counts are reported separately.
- Cancelled and returned labels are shown as status proxies and are not silently netted.
- Outliers are flagged and retained.
- Product joins are candidate SKU joins with match-rate disclosure.
- This notebook is descriptive and does not infer causation.